## מדידות חוזרות: ממוצע ושגיאת ממוצע לאורך ציר

במעבדה חוזרים על כל מדידה כמה פעמים, וכל חזרה נותנת תוצאה מעט שונה. מהחזרות מחשבים שלושה מספרים: את **הממוצע**, את **סטיית התקן המדגמית** (עד כמה מדידה בודדת מתפזרת סביב הממוצע) ואת **שגיאת הממוצע** (עד כמה הממוצע עצמו מדויק). הנוסחאות נתונות, והמשימה היא לכתוב אותן בקוד:

$$\bar x = \frac{1}{N}\sum_{i=1}^{N} x_i \qquad s = \sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(x_i-\bar x)^2} \qquad \sigma_{\bar x} = \frac{s}{\sqrt{N}}$$

כשהמדידות שמורות במערך בצורה (זווית, חזרה), כל אחד מהמספרים האלה הוא צמצום לאורך ציר החזרות, כמו בסעיף 9.6.

### דוגמה: ארבע חזרות בזווית אחת

ארבע מדידות של הטווח בזווית 45 מעלות, במטרים: 39, 39, 39 ו-43. את התוצאה אפשר לחשב בראש: הממוצע הוא 40, הסטיות ממנו הן ‎`-1, -1, -1, 3`‎, וסכום הריבועים שלהן הוא 12. לכן $s = \sqrt{12/3} = 2$, ושגיאת הממוצע היא $2/\sqrt{4} = 1$. הקוד כותב את שלוש הנוסחאות כפי שהן:

In [ ]:
import numpy as np

r45 = np.array([39.0, 39.0, 39.0, 43.0])   # range (m), four repetitions
N = len(r45)

mean = r45.sum() / N
s = np.sqrt(((r45 - mean)**2).sum() / (N - 1))
sem = s / np.sqrt(N)
print(mean, s, sem)

ל-NumPy יש גם מתודות לאותו חישוב: ‎`r45.mean()`‎ לממוצע, ו-‎`r45.std(ddof=1)`‎ לסטיית התקן המדגמית. הפרמטר ‎`ddof=1`‎ אומר לחלק ב-$N-1$, כמו בנוסחה; את שגיאת הממוצע מחשבים מסטיית התקן.

In [ ]:
print(r45.mean(), r45.std(ddof=1), r45.std(ddof=1) / np.sqrt(N))

### דוגמה: כל הזוויות לאורך ציר החזרות

עכשיו חמש זוויות, ארבע חזרות בכל אחת. כל שורה של ‎`R`‎ היא זווית, וכל עמודה היא חזרה. ‎`axis=1`‎ הוא ציר החזרות: הוא נעלם, ונשאר מספר אחד לכל זווית (כלל 2 מסעיף 9.1).

```{image} ../../figures/9_7_errorbar.svg
:alt: R.mean(axis=1) and the error of the mean give a point and an error bar per angle
:align: center
:class: only-light
```

```{image} ../../figures/9_7_errorbar-dark.svg
:alt: R.mean(axis=1) and the error of the mean give a point and an error bar per angle
:align: center
:class: only-dark
```

In [ ]:
angles_deg = np.array([15.0, 30.0, 45.0, 60.0, 75.0])
R = np.array([[19.4, 21.0, 20.2, 19.8],    # 15 degrees
              [34.1, 35.6, 34.9, 35.0],    # 30 degrees
              [39.0, 39.0, 39.0, 43.0],    # 45 degrees
              [35.8, 34.2, 35.5, 34.9],    # 60 degrees
              [20.5, 19.6, 20.1, 20.6]])   # 75 degrees: (angle, repetition)
print(R.shape)

N = R.shape[1]                       # repetitions per angle
means = R.mean(axis=1)               # shape (5,): one mean per angle
sem = R.std(axis=1, ddof=1) / np.sqrt(N)
print(means)
print(sem)

כמו בסעיף 8.8, מציגים את התוצאה כנקודות עם קווי שגיאה: הנקודה היא הממוצע, וקו השגיאה הוא שגיאת הממוצע. לצידן עקומת התיאוריה $R = v_0^2 \sin(2\theta)/g$, עם $v_0 = 20$ מ'/ש'.

In [ ]:
import matplotlib.pyplot as plt

g, v0 = 9.8, 20.0                    # m/s^2, m/s
angles_theory = np.linspace(0, 90, 200)
R_theory = v0**2 * np.sin(np.radians(2 * angles_theory)) / g

fig, ax = plt.subplots()
ax.plot(angles_theory, R_theory, color="gray", label="Theory")
ax.errorbar(angles_deg, means, yerr=sem, fmt="o", capsize=4,
            label="Mean ± error of the mean")
ax.set_xlabel("Launch angle (degrees)")
ax.set_ylabel("Range R (m)")
ax.legend()
plt.show()

### באג נפוץ: סטיית תקן בלי ddof

בברירת המחדל ‎`std`‎ מחלק ב-$N$ ולא ב-$N-1$. בשורה של 45 מעלות זה נותן 1.73 במקום 2, ולכן גם שגיאת ממוצע קטנה מדי. עם הרבה חזרות ההבדל קטן, אבל עם ארבע חזרות הוא כ-13%. שגיאה אין: הקוד רץ ומחזיר מספר סביר, ולכן קל לפספס את הבאג.

In [ ]:
print(r45.std())          # divides by N: wrong for a sample
print(r45.std(ddof=1))    # divides by N - 1: the formula above

### נסו בעצמכם

הטווח תלוי במהירות השיגור: $R = v_0^2 \sin(2\theta)/g$. אם למהירות יש שגיאה $\sigma_{v_0}$, השגיאה בטווח נתונה בנוסחה

$$\sigma_R = \left|\frac{dR}{dv_0}\right| \sigma_{v_0}, \qquad \frac{dR}{dv_0} = \frac{2 v_0 \sin(2\theta)}{g}$$

השלימו את ‎`range_error`‎ לפי החתימה. את הערך המוחלט מחשבת הפונקציה המובנית ‎`abs`‎, למשל ‎`abs(-2.0)`‎ שווה 2.0. השורה הראשונה בודקת אותה מול דוגמה שאפשר לחשב בראש. אחר כך חשבו את השגיאה בטווח בזווית 30 מעלות, עם $v_0 = 20$ מ'/ש', $\sigma_{v_0} = 0.4$ מ'/ש' ו-$g = 9.8$.

In [ ]:
def range_error(v0, sigma_v0, angle_deg, g):
    """Error (m) of the range R, from the error sigma_v0 (m/s) of v0."""
    pass   # replace with your code

# expected: 0.2  (v0 = 10, sigma_v0 = 0.1, 45 degrees, g = 10)
print(range_error(10.0, 0.1, 45.0, 10.0))

`````{admonition} פתרון
:class: dropdown, tip
```python
def range_error(v0, sigma_v0, angle_deg, g):
    """Error (m) of the range R, from the error sigma_v0 (m/s) of v0."""
    dR_dv0 = 2 * v0 * np.sin(np.radians(2 * angle_deg)) / g
    return abs(dR_dv0) * sigma_v0

print(range_error(10.0, 0.1, 45.0, 10.0))   # 0.2
print(range_error(20.0, 0.4, 30.0, 9.8))    # 1.4139...
```
בדוגמה: $dR/dv_0 = 2 \cdot 10 \cdot 1/10 = 2$, ולכן $\sigma_R = 2 \cdot 0.1 = 0.2$ מטר. הערך המוחלט שומר על שגיאה חיובית גם כשהנגזרת שלילית.
`````

### בדקו את עצמכם

שתי השאלות הראשונות מתייחסות לקוד הבא:

```python
Q = np.array([[1.0, 1.0, 1.0, 5.0],
              [2.0, 2.0, 2.0, 6.0]])
print(Q.mean(axis=1))          # line A
print(Q.std(axis=1, ddof=1))   # line B
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מה המספר האחרון ששורה A מדפיסה?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "3.0",
                "correct": true,
                "feedback": "נכון: בשורה השנייה הסכום הוא 12, וחלוקה ב-4 נותנת 3."
            },
            {
                "answer": "2.0",
                "correct": false,
                "feedback": "לא: 2 הוא הממוצע של השורה הראשונה, והמספר האחרון שייך לשורה השנייה."
            },
            {
                "answer": "6.0",
                "correct": false,
                "feedback": "לא: 6 הוא האיבר הגדול בשורה השנייה, לא הממוצע שלה."
            },
            {
                "answer": "2.5",
                "correct": false,
                "feedback": "לא: axis=1 מחשב ממוצע לכל שורה בנפרד, ולא את הממוצע של כל המערך."
            }
        ]
    },
    {
        "question": "מה המספר הראשון ששורה B מדפיסה?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "2.0",
                "correct": true,
                "feedback": "נכון: שלוש סטיות של מינוס 1 וסטייה אחת של 3 נותנות סכום ריבועים 12, ו-12 חלקי 3 הוא 4. השורש של 4 הוא 2."
            },
            {
                "answer": "1.73",
                "correct": false,
                "feedback": "לא: זו התוצאה של חלוקה ב-4 במקום ב-3, כלומר בלי ddof=1."
            },
            {
                "answer": "4.0",
                "correct": false,
                "feedback": "לא: 4 הוא הריבוע של סטיית התקן, לפני השורש."
            },
            {
                "answer": "1.0",
                "correct": false,
                "feedback": "לא: 1 היא שגיאת הממוצע, סטיית התקן חלקי השורש של 4."
            }
        ]
    },
    {
        "question": "חוזרים על מדידה 16 פעמים במקום 4, והפיזור של המדידות נשאר בערך אותו דבר. מה קורה לשגיאת הממוצע?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "קטנה פי 2",
                "correct": true,
                "feedback": "נכון: שגיאת הממוצע היא s חלקי השורש של N, והשורש של 16 גדול פי 2 מהשורש של 4."
            },
            {
                "answer": "קטנה פי 4",
                "correct": false,
                "feedback": "לא: היא תלויה בשורש של N, ולכן פי 4 יותר מדידות נותנים שגיאה קטנה רק פי 2."
            },
            {
                "answer": "לא משתנה",
                "correct": false,
                "feedback": "לא: s לא משתנה, אבל המכנה, השורש של N, גדל."
            },
            {
                "answer": "גדלה פי 2",
                "correct": false,
                "feedback": "לא: יותר מדידות נותנות ממוצע מדויק יותר, לא פחות."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)